# Replication notebook: *Aspect is not a fixed property of linguistic expressions*

White, Grimm, & Glass (2026), Experiments in Linguistic Meaning (ELM).

This notebook reproduces every model, measure, and figure in the paper. It is
organized so that each group of cells answers one question the paper asks, in
the order the paper asks it:

- §2 Experiment 1a (cloze): do event descriptions that share a verb and a
  syntactic frame differ in their preference for *in* over *for*?
- §3 Experiment 1b (temporal-overlap): does that preference survive the two
  known confounds of the *in*/*for* diagnostic?
- §4 Aspectual match: how do two model fits become a probability, for each
  pair of sentences, that the two match aspectually?
- §5 Experiment 2 (similarity): are two descriptions that match
  aspectually rated as more similar, over and above whether they share a sense?
- §6 Discussion: which verbs drive the effect?

All five sections use *per-sentence* quantities. Because the paper's claim
concerns variation below the sense, no quantity is a verb-level or sense-level
average.

Run the cells top to bottom. The `brms` fits cache to `analysis/fits/` and the
derived measures to `analysis/cache/`, so a second run resumes where the first
left off. A cold run takes roughly 9 to 11 hours on an 8-core M-series Mac, about half
of it the §5 fit.

## Setup

This cell loads the packages, sets the seed, anchors the working directory at
the repository root, and creates the cache and figure directories.

The top-level seed covers sampling outside Stan: the KDE, the draws subsampled
in §4, and the jitter in Figure 4. Each `brm` call has a separate `seed`, which
keeps its chains reproducible independently.

In [ ]:
suppressPackageStartupMessages({
  library(readr); library(dplyr); library(tidyr); library(stringr)
  library(brms); library(ordbetareg); library(posterior); library(cmdstanr)
  library(ggplot2); library(ggrepel); library(patchwork); library(betareg)
  library(MASS); library(showtext); library(scales)
})
set.seed(2030491)

# Anchor paths at repo root, regardless of where Jupyter launched.
if (basename(getwd()) == "analysis") setwd("..")
stopifnot(file.exists("data/data-cloze.csv"))

dir.create("analysis/fits",  showWarnings = FALSE, recursive = TRUE)
dir.create("analysis/cache", showWarnings = FALSE, recursive = TRUE)
dir.create("figures",        showWarnings = FALSE, recursive = TRUE)

# Inter font for figures; downloads from Google Fonts on first call.
sysfonts::font_add_google("Inter", "Inter")
showtext_opts(dpi = 300); showtext_auto()
theme_set(theme_bw(base_family = "Inter"))

# ---- Convergence diagnostics reporting (appendix) -------------------------
# Report the worst
# Rhat and smallest effective sample sizes across all parameters (fixed, random
# SDs, correlations, group-level deviations), the HMC pathologies (divergent
# transitions, max-treedepth saturation), and per-chain E-BFMI.
report_diagnostics <- function(fit, name, max_treedepth = 12) {
  dr    <- posterior::as_draws(fit)
  drw   <- posterior::summarise_draws(dr, "rhat", "ess_bulk", "ess_tail")
  np    <- brms::nuts_params(fit)
  ndiv  <- sum(np$Value[np$Parameter == "divergent__"])
  ntree <- sum(np$Value[np$Parameter == "treedepth__"] >= max_treedepth)
  en    <- np[np$Parameter == "energy__", ]
  ebfmi <- tapply(en$Value, en$Chain,
                  function(E) sum(diff(E)^2) / sum((E - mean(E))^2))
  cat(sprintf("\n=== %s: convergence diagnostics (%d post-warmup draws) ===\n",
              name, posterior::ndraws(dr)))
  cat(sprintf("  max Rhat             : %.4f  (want < 1.01)\n", max(drw$rhat, na.rm = TRUE)))
  cat(sprintf("  min Bulk-ESS         : %.0f\n", min(drw$ess_bulk, na.rm = TRUE)))
  cat(sprintf("  min Tail-ESS         : %.0f\n", min(drw$ess_tail, na.rm = TRUE)))
  cat(sprintf("  divergent transitions: %d\n", ndiv))
  cat(sprintf("  max-treedepth hits   : %d  (limit %d)\n", ntree, max_treedepth))
  cat(sprintf("  E-BFMI per chain     : %s  (want > 0.3)\n",
              paste(sprintf("%.2f", ebfmi), collapse = ", ")))
  invisible(drw)
}

## Load and prepare data

Each raw CSV under `data/` holds one of the three experiments, one row per
trial. We derive a few helper columns here and fix the factor codings once, so
that no later cell has to handle contrasts.

Two of those codings matter downstream. First, the generality factor is ordered
along the cline (*contentful* &prec; *bleached* &prec; *templatic*, §2.1) and
given successive-difference contrasts, so that its two coefficients estimate
per-step movement along the cline rather than comparisons against an arbitrary
baseline. Second, `log_minutes_table` maps each of the eleven time units to
log-minutes. That table serves two purposes: it supplies
`duration_in_log_minutes` for the per-sentence KDE mode in §4, and it is the
snap target for the canonical evaluation cell.

We work in log-minutes because the time units form a geometric series: eleven
steps span
roughly twelve orders of magnitude from milliseconds to millennia, so that on a
linear scale the long units tend to swamp the rest. On a log scale the steps
come out roughly evenly spaced, which is what the kernel density estimator in §4
requires.

In [ ]:
# ----- Experiment 1a: cloze (§2) -----
log_minutes_table <- c(
  milliseconds = -log(60) - log(1000),
  seconds      = -log(60),
  minutes      = 0,
  hours        = log(60),
  days         = log(60) + log(24),
  weeks        = log(60) + log(24) + log(7),
  months       = log(60) + log(24) + log(4.35 * 7),
  years        = log(60) + log(24) + log(4.35 * 7 * 12),
  decades      = log(60) + log(24) + log(4.35 * 7 * 12 * 10),
  centuries    = log(60) + log(24) + log(4.35 * 7 * 12 * 100),
  millenia     = log(60) + log(24) + log(4.35 * 7 * 12 * 1000)
)

data_cloze <- read_csv("data/data-cloze.csv", show_col_types = FALSE) %>%
  rename(generality = contentfulness) %>%
  mutate(
    # PropBank sense + concreteness label for contentful items; a synthetic
    # verb-level label for bleached and templatic items (the mm() weights
    # in the fit redistribute these onto the contentful labels).
    verb_sense = ifelse(
      generality == "contentful",
      paste(propbank_sense, concreteness, sep = "_"),
      paste(verb, generality, sep = "_")
    ),
    time_unit = factor(duration, levels = names(log_minutes_table)),
    duration  = paste(numeral, time_unit, sep = "_"),
    # Argument-shape factor: collapses the four binary plural/mass features
    # into one string so brms can absorb it as a grouping variable. Templatic
    # items get a single dedicated level.
    argument_forms = paste(
      ifelse(generality == "templatic", "subj:templatic",
             ifelse(subject_noun_plural, "subj:plural", "subj:singular")),
      ifelse(generality == "templatic", "obj:templatic",
             ifelse(object_noun_plural, "obj:plural", "obj:singular")),
      ifelse(generality == "templatic", "subj:templatic",
             ifelse(subject_noun_mass, "subj:mass", "subj:count")),
      ifelse(generality == "templatic", "obj:templatic",
             ifelse(object_noun_mass, "obj:mass", "obj:count")),
      sep = "_"
    ),
    listid = paste(generality, clause_type, list_num, sep = "_")
  )

data_cloze_episodic <- data_cloze %>%
  filter(clause_type == "episodic") %>%
  mutate(
    duration_in_log_minutes = log_minutes_table[as.character(time_unit)] +
      log(numeral),
    preposition = factor(preposition, levels = c("for", "in")),
    generality  = factor(generality,  levels = c("contentful", "bleached", "templatic")),
    verb_type   = factor(verb_type,   levels = c("manner", "result"))
  )
# Successive-difference contrasts on generality, per §2.4.
contrasts(data_cloze_episodic$generality) <- MASS::contr.sdif(3)
colnames(contrasts(data_cloze_episodic$generality)) <-
  c("bleached-contentful", "templatic-bleached")

cat(sprintf("cloze:      %d trials, %d sentences, %d verbs\n",
            nrow(data_cloze_episodic),
            n_distinct(data_cloze_episodic$sentence),
            n_distinct(data_cloze_episodic$verb)))

### Latent sense proportions

A bleached sentence such as *The thing hit the other thing* may be compatible
with several PropBank senses, and a templatic sentence such as *X hit Y* is
compatible with all of them. The sense under which a participant reads such a
sentence is not observed. Assigning the sentence one sense label would thus
distort the by-sense random effect.

We represent this uncertainty with a *multi-membership* (MM) effect, which lets
a single observation load onto several levels of a grouping factor at once.
Each item has a distribution over the (sense, concreteness) combinations of its
verb, and the effect averages over it:

- *Contentful* items place a point mass on the combination they were written
  for (§2.1).
- *Templatic* items are weighted by $p(\text{sense} \mid \text{verb})$.
- *Bleached* items are weighted by $p(\text{sense} \mid \text{verb})$ as well,
  except that every combination other than those of the contentful sentences
  the item was constructed from is discounted by a factor $\varepsilon$. The
  inventory `data/manner-result-sentences.csv` records which contentful
  sentences each bleached sentence was constructed from.

Neither $p(\text{sense} \mid \text{verb})$ nor $\varepsilon$ is fixed in
advance. They are parameters of each regression:

- For each verb, the proportions $\rho$ of its rolesets have a flat Dirichlet
  prior, and the verb's OntoNotes roleset counts are multinomial given $\rho$.
- OntoNotes does not distinguish concrete from abstract uses of a roleset. For
  each roleset with contentful sentences at both concreteness levels, the share
  that is concrete has a $\mathrm{Beta}(1, 1)$ prior.
- $\varepsilon \sim \mathrm{Beta}(1, 4)$.

brms's `mm()` takes its weights as data. We thus let brms generate each
regression's Stan program, edit that program so that the weights are computed
from these parameters (`edit_mm_stan()` in `analysis/src/latent_senses.R`),
sample it through `cmdstanr`, and read the draws back into the `brmsfit`. The
regression itself (its fixed effects, random effects, and priors) is exactly
the one `brm()` would fit.

The two regressions are fit in sequence, and the posterior over
$(\rho, \text{split}, \varepsilon)$ from the cloze fit (§2) is the prior for
the proportion fit (§3). The roleset counts thus enter the likelihood once, in
the cloze fit. The similarity analysis (§5) does not estimate these parameters;
it uses their posterior from the proportion fit. `carry_forward()` passes a posterior on by
matching moments: a Dirichlet for each verb's roleset proportions and a Beta
for each split and for $\varepsilon$.

The counts, in `data/propbank-sense-distribution.csv`, are per-verb
PropBank-roleset frequencies from the English portion of OntoNotes 5.0
(Weischedel et al. 2013, LDC2013T19). OntoNotes is LDC-licensed and cannot be
redistributed. The extraction script
`data/scripts/make_propbank_sense_distribution.py` rebuilds this file from a
local copy of the OntoNotes release tarball.

In [ ]:
source("analysis/src/latent_senses.R")

# Senses, rolesets, and each sentence's slots over the senses of its verb.
tab <- sense_tables()
K <- K_SLOTS
cat(sprintf("%d senses over %d rolesets (%d with both concreteness levels); %d sentences\n",
            nrow(tab$senses), nrow(tab$roles), tab$n_split, nrow(tab$items)))
# Columns verb_sense1..4 name the senses of each sentence's verb. The weight
# columns are placeholders: the weights are parameters of the regressions.
item_wide <- slot_columns(tab)

# Fit one regression with latent sense weights. `prior` is NULL for the first
# stage, or the previous stage's posterior from carry_forward().
fit_latent <- function(formula, data, family, name, seed, prior = NULL,
                       warmup = 2500, iter = 5000, adapt_delta = 0.99) {
  b  <- brm(formula, data = data, family = family, empty = TRUE)
  sd <- standata(b); class(sd) <- "list"
  # the senses must be numbered as brms numbers the levels of the mm() factor
  stopifnot(identical(align_senses(tab, sd, data)$senses$verb_sense, tab$senses$verb_sense))
  m <- sample_edited(b, edit_mm_stan(stancode(b)),
                     c(sd, sense_standata(tab, data$sentence, prior = prior)),
                     name = name, seed = seed, warmup = warmup, iter = iter,
                     adapt_delta = adapt_delta)
  with_mean_weights(m, tab)
}

## §2 Experiment 1a: A cloze diagnostic for telicity

Experiment 1a measures, for each sentence, how strongly speakers prefer a
temporal modifier headed by *in* over one headed by *for*, the classic telicity
diagnostic. The fit (§2.4) is a Bayesian mixed-effects logistic regression
predicting that choice from `verb_type` crossed with `generality`.

The random-effects structure absorbs the standard formal predictors of
telicity (argument shape, modifier form, sense, and the design factors), so
that residual variation at the by-`sentence` and by-`verb_sense` levels is not
attributed to them. The corresponding random-effect SDs quantify the paper's
first result. The
by-`verb_sense` intercept is the multi-membership effect described above,
so that bleached and templatic items contribute fractionally to each
(sense, concreteness) combination of their verb, with weights that are
estimated along with the regression.

The fit caches to `analysis/fits/cloze-latent.rds`; a cold run takes about an
hour.

In [ ]:
cloze_data <- data_cloze_episodic %>%
  left_join(item_wide, by = "sentence")
contrasts(cloze_data$verb_type) <- contr.sum(2)

f_cloze <-
  # Fixed effects: verb_type x generality interaction.
  preposition ~ verb_type * generality +
    # Maximal random-effects structure the design supports.
    (1 + verb_type | participant) +
    (1 + verb_type | listid) +
    (1 | sentence) +
    (1 | argument_forms) +
    (1 + generality | verb) +
    # Multi-membership sense effect; its weights are estimated.
    (1 | mm(verb_sense1, verb_sense2, verb_sense3, verb_sense4,
            weights = cbind(weight1, weight2, weight3, weight4),
            scale = FALSE)) +
    (1 | time_unit) +
    (1 | duration)

# Number the senses as brms numbers the levels of the mm() grouping factor.
tab <- align_senses(tab, standata(brm(f_cloze, data = cloze_data,
                                      family = bernoulli(), empty = TRUE)),
                    cloze_data)

cloze_fit_path <- "analysis/fits/cloze-latent.rds"
if (!file.exists(cloze_fit_path)) {
  m_cloze <- fit_latent(f_cloze, data = cloze_data, family = bernoulli(),
                        name = "cloze-latent", seed = 2030491)
  saveRDS(m_cloze, cloze_fit_path)
} else {
  m_cloze <- readRDS(cloze_fit_path)
}
print(round(summary(m_cloze)$fixed[, c("Estimate","l-95% CI","u-95% CI","Rhat","Bulk_ESS")], 3))
report_diagnostics(m_cloze, "§2 Cloze")

# The prior on the sense parameters for §3: their posterior from the cloze fit.
prior_prop <- carry_forward(sense_draws(m_cloze), tab)

### Figure 1 (§2.5 Results)

Cloze *in*-rates by `verb type` × `generality` on the left, with the per-group
random-effect SDs and their share of total variance on the right.

The by-sentence and by-verb-sense SDs in the right panel together exceed the
verb-class effect for many items. This item-level variability is the result
reported in §2.5.

In [ ]:
# Helper: per-grouping random-effect SD and proportion of variance.
re_sd_and_propvar <- function(model, residual_variance = pi^2 / 3,
                              probs = c(0.025, 0.975)) {
  draws  <- as_draws_df(model)
  groups <- names(VarCorr(model))
  group_var <- sapply(groups, function(g) {
    cols <- grep(paste0("^sd_", g, "__"), names(draws), value = TRUE)
    if (length(cols) == 0) return(rep(0, nrow(draws)))
    rowSums(as.matrix(draws[, cols, drop = FALSE])^2)
  })
  total_var <- rowSums(group_var) + residual_variance
  prop_var  <- group_var / total_var
  group_sd  <- sqrt(group_var)
  data.frame(
    group  = colnames(group_var),
    sd_med = apply(group_sd, 2, median),
    sd_lo  = apply(group_sd, 2, quantile, probs[1]),
    sd_hi  = apply(group_sd, 2, quantile, probs[2]),
    pv_med = apply(prop_var, 2, median),
    pv_lo  = apply(prop_var, 2, quantile, probs[1]),
    pv_hi  = apply(prop_var, 2, quantile, probs[2]),
    row.names = NULL
  )
}

# Bar chart of per-group SDs (left axis) + proportion of variance (right axis).
re_bar_chart <- function(re_df, labels, palette_fill = "#4575b4") {
  d <- re_df[order(-re_df$sd_med), ]
  d$group <- factor(unname(labels[d$group]), levels = unname(labels[d$group]))
  scale_factor <- max(d$sd_hi) / max(d$pv_hi)
  dodge_w <- 0.5
  ggplot(d, aes(x = group)) +
    geom_hline(yintercept = 0, color = "black") +
    geom_errorbar(aes(ymin = sd_lo, ymax = sd_hi),
                  position = position_nudge(x = -dodge_w / 2), width = 0.15) +
    geom_point(aes(y = sd_med),
               position = position_nudge(x = -dodge_w / 2),
               shape = 21, size = 4, fill = palette_fill, color = "black") +
    geom_errorbar(aes(ymin = pv_lo * scale_factor, ymax = pv_hi * scale_factor),
                  position = position_nudge(x = dodge_w / 2),
                  width = 0.15, linetype = "dashed") +
    geom_point(aes(y = pv_med * scale_factor),
               position = position_nudge(x = dodge_w / 2),
               shape = 23, size = 4, fill = palette_fill, color = "black") +
    scale_y_continuous(
      name = "Random effect SD",
      sec.axis = sec_axis(~ . / scale_factor,
        name = "Proportion of variance",
        labels = scales::percent_format(accuracy = 1))
    ) +
    xlab("") +
    theme(legend.position = "none",
          axis.text   = element_text(size = 18, color = "black"),
          axis.text.x = element_text(angle = 45, hjust = 1),
          axis.title  = element_text(size = 22, face = "bold"))
}

# Empirical in-rates by cell, plus per-verb dots.
prep_props <- data_cloze_episodic %>%
  group_by(verb_type, generality) %>%
  summarize(prop_in = mean(preposition == "in"), .groups = "drop")
prep_props_by_verb <- data_cloze_episodic %>%
  group_by(verb_type, verb, generality) %>%
  summarize(prop_in = mean(preposition == "in"), .groups = "drop")

p_cloze_main <- ggplot() +
  geom_point(data = prep_props_by_verb, aes(x = verb_type, y = prop_in)) +
  geom_point(data = prep_props,
             aes(x = verb_type, y = prop_in), size = 5, color = "black") +
  geom_point(data = prep_props,
             aes(x = verb_type, y = prop_in, color = verb_type), size = 4) +
  geom_label_repel(data = prep_props_by_verb,
                   aes(x = verb_type, y = prop_in, label = verb, fill = verb_type),
                   max.overlaps = 20, size = 4.5, fontface = "bold") +
  facet_grid(~ generality) +
  scale_y_continuous(expression(bold("Proportion ") * bolditalic("in") * bold(" responses")),
                     limits = c(0, 1)) +
  xlab("") +
  scale_fill_brewer(type = "qual", palette = 4) +
  scale_color_brewer(type = "qual", palette = 4) +
  theme(legend.position = "none",
        strip.text  = element_text(size = 18, face = "bold"),
        axis.text   = element_text(size = 18, color = "black"),
        axis.title  = element_text(size = 22, face = "bold"))

cloze_labels <- c(
  participant = "particip.", listid = "list", sentence = "sent.",
  argument_forms = "arg. forms", verb = "verb",
  mmverb_sense1verb_sense2verb_sense3verb_sense4 = "verb sense",
  time_unit = "time unit", duration = "dur."
)
re_df_cloze <- re_sd_and_propvar(m_cloze)
p_re_cloze  <- re_bar_chart(re_df_cloze, cloze_labels)

p_combined_cloze <- p_cloze_main + p_re_cloze + plot_layout(widths = c(2.4, 1))
ggsave("figures/cloze_proportions_by_verb.pdf", p_combined_cloze,
       width = 18, height = 7.5, dpi = 300)
ggsave("figures/cloze_proportions_by_verb.png", p_combined_cloze,
       width = 18, height = 7.5, dpi = 300)
print(p_combined_cloze)

## §3 Experiment 1b: A temporal-overlap diagnostic for durativity

The *in*/*for* diagnostic has two known confounds: *in* can describe a delay before
the event starts (*The water was bubbling in two minutes*), and *for* can
describe how long a result state lasts (*The woman opened the window for five
minutes*). Both are non-durative in the relevant sense, and both tend to
contaminate the cloze measure. Experiment 1b addresses them by asking
participants directly what proportion of the modifier's span the event was
occurring.

The fit (§3.4) adds preposition preference (the per-sentence cloze *in*-rate
from §2, scaled to $[-1, 1]$) as a fixed effect interacting with `preposition`
and `generality`. The multi-membership sense effect is the same one used in the
cloze fit, and the prior on its sense parameters is their posterior from that
fit.

The quantity of interest is the `pref_in` × `preposition` interaction. If the
cloze preferences track event overlap, then sentences preferring *in* should
show more overlap under *in*, and sentences preferring *for* should show more
overlap under *for*. This interaction serves to validate the cloze measure; it
is not an independent hypothesis test.

The fit caches to `analysis/fits/proportion-latent.rds`; a cold run takes about three and a
half hours.

In [ ]:
# Per-sentence cloze in-rate, used as the pref_in predictor for the
# proportion fit (Section 3.4).
prep_props_by_sentence <- data_cloze_episodic %>%
  group_by(sentence) %>%
  dplyr::count(preposition) %>%
  dplyr::mutate(prop = n / sum(n), pref = n / sum(n) * 2 - 1) %>%
  pivot_wider(id_cols = sentence, names_from = preposition,
              values_from = c(prop, pref), values_fill = 0)

prop_raw <- read_csv("data/data-proportion.csv", show_col_types = FALSE) %>%
  rename(generality = contentfulness) %>%
  merge(prep_props_by_sentence, by = "sentence")

prop_data <- prop_raw %>%
  mutate(
    time_unit = factor(duration, levels = names(log_minutes_table)),
    duration  = paste(numeral, time_unit, sep = "_"),
    argument_forms = paste(
      ifelse(generality == "templatic", "subj:templatic",
             ifelse(subject_noun_plural, "subj:plural", "subj:singular")),
      ifelse(generality == "templatic", "obj:templatic",
             ifelse(object_noun_plural, "obj:plural", "obj:singular")),
      ifelse(generality == "templatic", "subj:templatic",
             ifelse(subject_noun_mass, "subj:mass", "subj:count")),
      ifelse(generality == "templatic", "obj:templatic",
             ifelse(object_noun_mass, "obj:mass", "obj:count")),
      sep = "_"
    ),
    listid = paste(generality, clause_type, list_num, sep = "_"),
    preposition = factor(preposition, levels = c("for", "in")),
    generality  = factor(generality,  levels = c("contentful", "bleached", "templatic")),
    verb_type   = factor(verb_type,   levels = c("manner", "result")),
    # Binarized overlap: 1 if participant chose any non-zero-overlap option.
    overlap = response != 0
  ) %>%
  left_join(item_wide, by = "sentence")
contrasts(prop_data$generality) <- MASS::contr.sdif(3)
colnames(contrasts(prop_data$generality)) <-
  c("bleached-contentful", "templatic-bleached")
# Preposition contrast: for = -1, in = +1, so the sign of pref_in:preposition
# is positive when sentences that prefer 'in' show more overlap under 'in'.
contrasts(prop_data$preposition) <- -contr.sum(2)

f_prop <-
  # Three-way interaction of preposition preference, preposition, and
  # generality (§3.4).
  overlap ~ pref_in * preposition * generality +
    (1 + pref_in * preposition | participant) +
    (1 + pref_in * preposition | listid) +
    (1 + preposition | sentence) +
    (1 + preposition * generality | verb) +
    (1 + preposition | mm(verb_sense1, verb_sense2, verb_sense3, verb_sense4,
                          weights = cbind(weight1, weight2, weight3, weight4),
                          scale = FALSE)) +
    (1 + preposition | argument_forms) +
    (1 | time_unit) +
    (1 | duration)

prop_fit_path <- "analysis/fits/proportion-latent.rds"
if (!file.exists(prop_fit_path)) {
  # The sense parameters' prior is their posterior from the cloze fit.
  m_prop <- fit_latent(f_prop, data = prop_data, family = bernoulli(),
                       name = "proportion-latent", seed = 2030490,
                       prior = prior_prop)
  saveRDS(m_prop, prop_fit_path)
} else {
  m_prop <- readRDS(prop_fit_path)
}
print(round(summary(m_prop)$fixed[, c("Estimate","l-95% CI","u-95% CI","Rhat","Bulk_ESS")], 3))
report_diagnostics(m_prop, "§3 Proportion")


### Figure 2 (§3.5 Results)

Per-verb non-overlap rates by preposition, with per-group random-effect SDs on
the right.

Four of the five response options indicate some overlap, so the overall
non-overlap rate is low by construction. The match score in §4 thus asks where a
sentence's overlap logit sits relative to those of the other sentences, not
what the level itself is.

In [ ]:
nonov_by_verb <- prop_data %>%
  group_by(verb_type, verb, preposition) %>%
  summarize(nonoverlap = mean(!overlap), .groups = "drop")
nonov_by_type <- prop_data %>%
  group_by(verb_type, preposition) %>%
  summarize(n_nonov = sum(!overlap), n_total = n(), .groups = "drop") %>%
  mutate(nonoverlap = n_nonov / n_total)
# Wilson-style CI on the cell rate via prop.test; correct = FALSE gives the
# plain Wilson score interval, which stays inside [0, 1] near the boundary.
ci <- mapply(function(x, n) {
  t <- prop.test(x, n, conf.level = 0.95, correct = FALSE)
  c(lo = t$conf.int[1], hi = t$conf.int[2])
}, nonov_by_type$n_nonov, nonov_by_type$n_total)
nonov_by_type$lo <- ci["lo", ]; nonov_by_type$hi <- ci["hi", ]

p_nonov_main <- ggplot() +
  geom_point(data = nonov_by_verb, aes(x = verb_type, y = nonoverlap),
             position = position_jitter(width = 0.15, height = 0, seed = 1)) +
  geom_linerange(data = nonov_by_type,
                 aes(x = verb_type, ymin = lo, ymax = hi), linewidth = 1.2) +
  geom_point(data = nonov_by_type, aes(x = verb_type, y = nonoverlap),
             size = 5, color = "black") +
  geom_point(data = nonov_by_type,
             aes(x = verb_type, y = nonoverlap, color = verb_type), size = 4) +
  geom_label_repel(data = nonov_by_verb,
                   aes(x = verb_type, y = nonoverlap, label = verb,
                       fill = verb_type),
                   max.overlaps = 20, size = 4.5, fontface = "bold") +
  facet_grid(~ preposition) +
  scale_y_continuous("Proportion non-overlap") +
  xlab("") +
  scale_fill_brewer(type = "qual", palette = 4) +
  scale_color_brewer(type = "qual", palette = 4) +
  theme(legend.position = "none",
        strip.text  = element_text(size = 18, face = "bold"),
        axis.text   = element_text(size = 18, color = "black"),
        axis.title  = element_text(size = 22, face = "bold"))

prop_labels <- c(
  participant = "particip.", listid = "list", sentence = "sent.",
  argument_forms = "arg. forms", verb = "verb",
  mmverb_sense1verb_sense2verb_sense3verb_sense4 = "verb sense",
  time_unit = "time unit", duration = "dur."
)
re_df_prop <- re_sd_and_propvar(m_prop)
p_re_prop  <- re_bar_chart(re_df_prop, prop_labels)

p_combined_prop <- p_nonov_main + p_re_prop + plot_layout(widths = c(2.4, 1))
ggsave("figures/nonoverlap_by_verb.pdf", p_combined_prop,
       width = 18, height = 7.5, dpi = 300)
ggsave("figures/nonoverlap_by_verb.png", p_combined_prop,
       width = 18, height = 7.5, dpi = 300)
print(p_combined_prop)

## §4 Aspectual match

The similarity analysis in §5 asks whether two descriptions that are alike
aspectually are judged more alike in meaning. It thus requires, for each pair
of sentences, a measure of how alike the two are aspectually. The earlier
experiments instead provide two regressions.

Each regression gives every sentence a position on the logit scale: the cloze
fit gives the log-odds of choosing *in* over *for*, and the proportion fit gives
the log-odds of an overlap response under *in* and under *for*. This section
turns the posterior over those three logits into a single per-pair probability.

The logits are defined only relative to a duration, and plausible durations vary
widely across the materials. *The batter hit the ball* and *The detective
investigated the crime* do not occupy the same time scale. We thus evaluate
each sentence at the time unit where its own empirical responses cluster.

Identifying that time unit takes four steps: convert each cloze response's
(numeral, time unit) pair to a duration in minutes, map that to log-minutes, run
a Gaussian kernel density estimator over the per-sentence log-minute durations
and take the mode, and snap to the time unit whose canonical
$(\text{numeral} = 1, t)$ duration sits closest to that mode.

We use a kernel density estimator because participants vary both the numeral
and the unit, and the two trade
off: *90 seconds* and *2 minutes* record nearly the same duration under
different units. The modal unit discards that agreement and can land on a unit
no single response supports, whereas smoothing in log-minutes lets the numeral
and the unit contribute
jointly.

In [ ]:
# Per-sentence modal time unit t_i (§4): a KDE on the log-minute duration
# distribution, snapped to the nearest canonical (1, t) cell. bw = "nrd0" is
# R's default Silverman bandwidth. The rationale is in the markdown above.
kde_mode <- function(x) {
  if (length(x) < 2) return(x[1])
  d <- density(x, bw = "nrd0", n = 2048); d$x[which.max(d$y)]
}
per_sent_eval <- data_cloze_episodic %>%
  group_by(verb, sentence) %>%
  summarize(mode_logmin = kde_mode(duration_in_log_minutes), .groups = "drop") %>%
  rowwise() %>%
  mutate(eval_time_unit = names(log_minutes_table)[
    which.min(abs(log_minutes_table - mode_logmin))
  ]) %>%
  ungroup()
print(per_sent_eval %>% count(eval_time_unit, sort = TRUE))

# §4 split: share of sentences whose evaluation t_i is <= days vs. >= weeks.
.short <- c("milliseconds","seconds","minutes","hours","days")
cat(sprintf("eval t_i <= days: %.0f%%; >= weeks: %.0f%%\n",
            100 * mean(per_sent_eval$eval_time_unit %in% .short),
            100 * mean(!per_sent_eval$eval_time_unit %in% .short)))

# Per-sentence cloze 'in'-rate on [-1, +1]: the pref_in predictor of §3.
prep_pref <- data_cloze_episodic %>%
  group_by(sentence) %>%
  summarize(pref_in = mean(preposition == "in") * 2 - 1, .groups = "drop")

### Per-sentence logits and reference-sentence match

**Logits.** For sentence $i$ and each posterior draw $s$ of the regressions,
let $x^{(s)}_i$ be the vector of three logits: choosing *in* in the cloze task,
and an overlap response under *in* and under *for* in the proportion task. Each
is the regression's linear predictor for an average participant and list, at
numeral $= 1$ and time unit $= t_i$ (the KDE mode from the previous cell). The
multi-membership sense effect is added draw by draw, with that draw's sense
weights, so that uncertainty about the sense of a bleached or templatic
sentence enters its logits. The two regressions are separate fits, so their
draws are independent and are paired by index.

**Why not agreement between responses.** The most direct match score would be
the probability that the two sentences receive the same response on each
diagnostic, $p_i p_j + (1 - p_i)(1 - p_j)$. This does not work for the overlap
diagnostics: four of the five response options indicate overlap, so nearly
every sentence has an overlap probability above $.9$ and nearly every pair
agrees. What distinguishes sentences is where they sit relative to one another.

**Reference sentences.** We thus compare the two sentences of a pair against
the other sentences in the materials. A reference sentence $k$ *separates*
sentences $i$ and $j$ on a logit if its own logit lies between theirs. On
posterior draw $s$, with $d$ ranging over the three logits,

$$\mathrm{sep}^{(s)}_{ijk} = \mathbb{1}\Bigl[\exists d:\ \min\bigl(x^{(s)}_{id}, x^{(s)}_{jd}\bigr) < x^{(s)}_{kd} < \max\bigl(x^{(s)}_{id}, x^{(s)}_{jd}\bigr)\Bigr].$$

The per-sentence posterior mean logits cache to
`analysis/cache/sentence_measures.rds`.


In [ ]:
sent_path <- "analysis/cache/sentence_measures.rds"
pair_path <- "analysis/cache/pair_measures.rds"

# Same-verb pairs from the similarity battery; we drop calibration items.
sim_raw <- read_csv("data/data-similarity.csv", show_col_types = FALSE) %>%
  filter(verb_type != "calibration")

if (!file.exists(sent_path) || !file.exists(pair_path)) {
  # Posterior draws of each sentence's three logits (draws x sentences x 3).
  logits <- sentence_logits(m_cloze, m_prop, tab, per_sent_eval, prep_pref)
  DIAG   <- dimnames(logits$x)[[3]]

  # Per sentence: posterior mean logits (for Figure 3).
  sentence_measures <- tibble(sentence = logits$sentence, verb = logits$verb,
                              generality = logits$generality) %>%
    bind_cols(setNames(as_tibble(apply(logits$x, c(2, 3), mean)), DIAG))
  saveRDS(sentence_measures, sent_path)

  # Per pair: the probability, over reference sentences and posterior draws,
  # that a reference does not separate the two sentences on any logit.
  pairs <- sim_raw %>% distinct(sentence1, sentence2, verb) %>%
    mutate(i = match(sentence1, logits$sentence), j = match(sentence2, logits$sentence))
  stopifnot(!anyNA(pairs$i), !anyNA(pairs$j))
  pairs$aspect_match <- sapply(seq_len(nrow(pairs)), function(p)
    reference_match(logits$x, pairs$i[p], pairs$j[p]))
  saveRDS(pairs, pair_path)
  rm(logits)
} else {
  sentence_measures <- readRDS(sent_path)
  pairs <- readRDS(pair_path)
}


### Per-pair aspectual match

§5 requires a per-*pair* quantity. For a pair $(i, j)$, aspectual match is the
probability that a reference sentence drawn at random from the other $N - 2$
sentences does not separate the two on any of the three logits, averaged over
posterior draws:

$$\mathrm{match}_{ij} = 1 - \frac{1}{S\,(N - 2)}\sum_{s=1}^{S}\ \sum_{k \neq i, j} \mathrm{sep}^{(s)}_{ijk}.$$

On a single logit, this is one minus the share of sentences that lie between
the two, i.e. one minus the distance between their quantiles among the
materials. Across the three logits, it is one minus the share of sentences
that lie between the two on at least one.

Two properties of the match are useful. First, it uses only the order of the
sentences on each logit. It is thus unchanged by any monotone transformation of
a logit, so that the high overall level of the overlap probabilities does not
affect it, and the three logits never have to be weighted against one another.
Second, computing it on each draw and then averaging carries the regressions'
uncertainty into the match, including the dependence between two sentences that
share a verb or a sense.

The match is relative to the materials: how far apart two sentences are is
measured by how many of the sentences in this study lie between them.


In [ ]:
# Attach the match scores back to each trial in the similarity battery.
pm <- sim_raw %>%
  left_join(pairs %>% dplyr::select(sentence1, sentence2, aspect_match),
            by = c("sentence1", "sentence2")) %>%
  mutate(similarity    = response / 100,
         sentence_pair = paste(pmin(sentence1, sentence2),
                               pmax(sentence1, sentence2), sep = "|"))
cat(sprintf("%d trials, %d unique pairs; aspectual match mean %.2f, SD %.2f\n",
            nrow(pm), n_distinct(pm$sentence_pair),
            mean(pm$aspect_match), sd(pm$aspect_match)))


### Figure 3 (§4)

The cell below first lists the pairs of contentful sentences with the highest
and lowest aspectual match, as a check on face validity.

The figure plots every contentful and bleached item by its probability of *in*
in the cloze task and its probability of non-overlap under *in*, faceted by
verb (manner first, then result, alphabetical within each block). The axes are
scaled in logits and labelled with the corresponding probabilities. A verb
with a tight hull has sentences that match one another aspectually; a verb
with a diffuse hull shows the within-verb variation that the paper is
concerned with.


In [ ]:
# Face validity (§4): the pairs of contentful sentences with the highest and
# lowest aspectual match.
cont <- sentence_measures %>% filter(generality == "contentful")
cc <- pairs %>% filter(sentence1 %in% cont$sentence, sentence2 %in% cont$sentence)
cat("\nhighest aspectual match:\n")
top <- cc %>% arrange(desc(aspect_match)) %>% head(6)
cat(sprintf("  %.2f  %s / %s\n", top$aspect_match, top$sentence1, top$sentence2), sep = "")
cat("lowest aspectual match:\n")
bot <- cc %>% arrange(aspect_match) %>% head(6)
cat(sprintf("  %.2f  %s / %s\n", bot$aspect_match, bot$sentence1, bot$sentence2), sep = "")

meta <- data_cloze_episodic %>% distinct(sentence, verb, verb_type)
# Drop templatic items: the similarity battery does not include them.
d_aspect <- inner_join(sentence_measures, meta, by = c("sentence", "verb")) %>%
  filter(generality != "templatic") %>%
  mutate(nonoverlap_in = -overlap_in)
d_aspect$verb_type <- factor(d_aspect$verb_type, levels = c("manner", "result"))
verb_order <- d_aspect %>% distinct(verb, verb_type) %>%
  arrange(verb_type, verb) %>% pull(verb)
d_aspect$verb <- factor(d_aspect$verb, levels = verb_order)
hull <- d_aspect %>% group_by(verb, verb_type) %>%
  slice(chull(cloze, nonoverlap_in)) %>% ungroup()

# The axes keep their logit scaling but are labelled with the corresponding
# probabilities. The y-axis is the non-overlap logit (the negated overlap
# logit), to match Figure 2.
p_x <- c(0.01, 0.1, 0.5, 0.9, 0.99)
p_y <- c(0.003, 0.01, 0.03, 0.1, 0.25, 0.5)
p_aspect <- ggplot(d_aspect, aes(x = cloze, y = nonoverlap_in,
                                 color = verb_type, fill = verb_type)) +
  geom_polygon(data = hull, alpha = 0.45, color = NA) +
  geom_point(alpha = 1.0, size = 1.2) +
  facet_wrap(~ verb, ncol = 10) +
  scale_color_brewer(name = "Verb type", type = "qual", palette = 4) +
  scale_fill_brewer( name = "Verb type", type = "qual", palette = 4) +
  scale_x_continuous(expression(bold("Probability of ") * bolditalic("in")),
                     breaks = qlogis(p_x), labels = p_x) +
  scale_y_continuous(expression(bold("Probability of non-overlap under ") * bolditalic("in")),
                     breaks = qlogis(p_y), labels = p_y) +
  theme(legend.position = "bottom",
        strip.text  = element_text(size = 14, face = "bold"),
        axis.text   = element_text(size = 12, color = "black"),
        axis.text.x = element_text(angle = 45, hjust = 1),
        axis.title  = element_text(size = 22, face = "bold"),
        legend.title = element_text(size = 18, face = "bold"),
        legend.text  = element_text(size = 14, color = "black"))
# Sized to match the other paper figures: width 18 in / 10 columns is about
# 1.65 in per panel, so the 4 rows of panels need ~6.6 in, plus ~2 in for
# the x-axis labels and the legend, which gives the 8.6 in height.
ggsave("figures/aspect_plane.pdf", p_aspect, width = 18, height = 8.6, dpi = 300)
ggsave("figures/aspect_plane.png", p_aspect, width = 18, height = 8.6, dpi = 300)
print(p_aspect)

## §5 Experiment 2: Sense similarity

Experiment 2 asks whether the variation measured in Experiments 1a and 1b is
reflected in judgments of similarity. The fit
(§5.4) is a Bayesian ordered-beta regression on the similarity rating, with
three pair-level predictors and all three of their pairwise interactions:

- aspectual match $\mathrm{match}_{ij}$, from §4;
- same-sense $s_{ij}$, the probability that the two items have the same
  PropBank roleset under their sense weights, with concreteness marginalized;
- generality pair $g_{ij}$, sum-coded (contentful-bleached v.
  contentful-contentful; the similarity battery contains no templatic items).

These predictors test four hypotheses from §5.3:

- **H1**: aspectual match predicts perceived similarity after controlling for
  sense identity.
- **H2**: pairs likely to share a sense are judged more similar, even after
  aspectual match is controlled for.
- **H3**: the contribution of aspectual match is *amplified* within a sense,
  which appears as an aspectual-match-by-same-sense interaction.
- **H4**: position on the generality cline shifts perceived similarity.

The analysis retains the slider's coding, so **lower values mean greater
similarity**. A hypothesis predicting greater similarity thus predicts a
*negative* coefficient.

Same-sense is computed in the same way as aspectual match: on each posterior
draw of the proportion fit, we take the probability that the two sentences
have the same roleset under that draw's sense weights, and we average over
draws (`same_sense_prob()` in `analysis/src/latent_senses.R`). Both predictors
are thus posterior expectations from Experiments 1a and 1b, and the similarity
ratings inform neither.

The fit caches to `analysis/fits/similarity.rds`; a cold run takes several
hours.


In [ ]:
# Same-sense: the probability that the two sentences have the same roleset,
# averaged over the posterior draws of the proportion fit's sense weights.
pair_ss <- pairs %>%
  mutate(p_same_sense = same_sense_prob(m_prop, tab, sentence1, sentence2)) %>%
  dplyr::select(sentence1, sentence2, p_same_sense)

# Standardize the two continuous predictors; sum-code generality pair.
pm <- pm %>%
  left_join(pair_ss, by = c("sentence1", "sentence2")) %>%
  mutate(aspect_match_c = as.numeric(scale(aspect_match)),
         same_sense_c  = as.numeric(scale(p_same_sense)),
         genpair = factor(paste(pmin(contentfulness1, contentfulness2),
                                pmax(contentfulness1, contentfulness2),
                                sep = "-")))
contrasts(pm$genpair) <- contr.sum(2)
colnames(contrasts(pm$genpair)) <- "CBvCC"
cat(sprintf("similarity trials: %d (%d unique pairs)\n",
            nrow(pm), n_distinct(pm$sentence_pair)))

### §5.4 Analysis: ordered-beta regression

The response is a slider value in $[0, 1]$ with substantial mass at both
endpoints, since participants do use "completely identical" and "extremely
different". This rules out ordinary beta regression, which assigns density zero
to exactly 0 and exactly 1. *Ordered beta* regression (Kubinec 2023)
accommodates it by modeling the two endpoints as discrete outcomes alongside a
beta density for the interior, which is why the fit goes through `ordbetareg`
rather than `brms` directly.

For the main effects we use the maximal random-effects structure the design
supports (by-`verb` and by-`participant` intercepts and slopes for each
main-effect predictor, plus a by-`sentence_pair` intercept), following Barr et al. (2013).

The cell also prints the summary the paper reports: the posterior mean and 95%
CI of each fixed effect.

In [ ]:
sim_fit_path <- "analysis/fits/similarity.rds"
# The ^2 expands to the three main effects plus all three pairwise
# interactions; H3 is the aspectual-match-by-same-sense one. The formula is
# `sim_formula` in analysis/src/latent_senses.R:
#   similarity ~ (aspect_match_c + same_sense_c + genpair)^2 +
#     (1 + aspect_match_c + same_sense_c + genpair | verb) +
#     (1 + aspect_match_c + same_sense_c + genpair | participant) +
#     (1 | sentence_pair)
if (!file.exists(sim_fit_path)) {
  b_sim <- ordbetareg(formula = sim_formula, data = pm, empty = TRUE,
                      backend = "cmdstanr")
  sd_sim <- standata(b_sim); class(sd_sim) <- "list"
  # sample_edited() runs a brms program through cmdstanr and reads the result
  # back as a brmsfit; the program is not edited here.
  m_sim <- sample_edited(b_sim, stancode(b_sim), sd_sim, name = "similarity",
                         seed = 2030492, warmup = 2500, iter = 5000,
                         adapt_delta = 0.995)
  saveRDS(m_sim, sim_fit_path)
} else {
  m_sim <- readRDS(sim_fit_path)
}

# §5.5 Results: posterior mean and 95% CI of each fixed effect.
dr <- as_draws_df(m_sim)
fx <- grep("^b_", colnames(dr), value = TRUE)
fx_tbl <- do.call(rbind, lapply(fx, function(v) {
  x <- dr[[v]]
  data.frame(coef  = sub("^b_", "", v),
             mean  = mean(x),
             lo95  = quantile(x, 0.025, names = FALSE),
             hi95  = quantile(x, 0.975, names = FALSE))
}))
print(fx_tbl, row.names = FALSE, digits = 3)
report_diagnostics(m_sim, "§5 Similarity")


### Figure 4 (§5.5 Results)

Per-pair mean dissimilarity against aspectual match, colored by the pair's
same-sense probability. The lines are the ordered-beta model's expected rating
at same-sense probability 0 and 1, for each generality pair, with 95% credible
intervals; six illustrative pairs are labeled.

H3 appears here as a *difference in slope*: the line for pairs that share a
sense falls more steeply with aspectual match than the line for pairs that do
not.


In [ ]:
# One row per pair: mean rating, aspectual match, and same-sense probability.
pair_means <- pm %>%
  mutate(contpair = as.character(genpair)) %>%
  group_by(sentence_pair, sentence1, sentence2, contpair) %>%
  summarize(similarity   = mean(similarity),
            aspect_match = first(aspect_match),
            p_same_sense = first(p_same_sense), .groups = "drop")

# X-axis jitter (±0.05) on the match score to relieve overplotting,
# clipped to [0, 1]. Only x is jittered; y is the mean rating.
set.seed(42)
.clip_jitter <- function(x, w = 0.05) pmin(1, pmax(0, x + runif(length(x), -w, w)))
pair_means$aspect_match_j <- .clip_jitter(pair_means$aspect_match)

# Lines: the ordered-beta model's expected rating as a function of aspectual
# match, for an average verb, participant, and pair, at same-sense
# probability 0 and 1 and for each generality pair. The expected rating is
# P(1) + P(interior) * plogis(eta), with the two endpoint probabilities given
# by the model's cutpoints. Bands are 95% credible intervals.
model_lines <- function(ngrid = 100) {
  dr <- as_draws_df(m_sim)
  g <- expand.grid(x = seq(min(pm$aspect_match), max(pm$aspect_match), length.out = ngrid),
                   p_same_sense = c(0, 1),
                   contpair = levels(pm$genpair), stringsAsFactors = FALSE)
  am <- (g$x - mean(pm$aspect_match)) / sd(pm$aspect_match)
  ss <- (g$p_same_sense - mean(pm$p_same_sense)) / sd(pm$p_same_sense)
  gp <- contrasts(pm$genpair)[g$contpair, "CBvCC"]
  X <- cbind(1, am, ss, gp, am * ss, am * gp, ss * gp)
  B <- as.matrix(dr[, c("b_Intercept", "b_aspect_match_c", "b_same_sense_c",
                        "b_genpairCBvCC", "b_aspect_match_c:same_sense_c",
                        "b_aspect_match_c:genpairCBvCC",
                        "b_same_sense_c:genpairCBvCC")])
  eta <- B %*% t(X)                                  # draws x grid
  c0 <- dr$cutzero; c1 <- dr$cutzero + exp(dr$cutone)
  p_one <- plogis(eta - c1); p_mid <- plogis(eta - c0) - p_one
  ev <- p_one + p_mid * plogis(eta)
  cbind(g, y = colMeans(ev),
        lo = apply(ev, 2, quantile, 0.025), hi = apply(ev, 2, quantile, 0.975))
}
smooth_match <- model_lines()

# Six focal pairs to label on the scatter.
vlab <- function(s, v, o) sprintf("%s %s %s", s, toupper(v), o)
focal_pairs <- list(
  list("The consultant opened the discussion.", "The person opened the thing.",
       paste(vlab("consultant","open","discussion"),
             vlab("person","open","thing"), sep="\n")),
  list("The cat killed the mouse.", "The joke killed the evening.",
       paste(vlab("cat","kill","mouse"), vlab("joke","kill","evening"), sep="\n")),
  list("The couple killed the evening.", "The traveler killed the afternoon.",
       paste(vlab("couple","kill","evening"),
             vlab("traveler","kill","afternoon"), sep="\n")),
  list("The cat killed the mouse.", "The criminal killed the victim.",
       paste(vlab("cat","kill","mouse"), vlab("criminal","kill","victim"), sep="\n")),
  list("The explorer discovered the island.", "The burglar discovered the key.",
       paste(vlab("explorer","discover","island"),
             vlab("burglar","discover","key"), sep="\n")),
  list("The robber shattered the window.", "The things shattered the other thing.",
       paste(vlab("robber","shatter","window"),
             vlab("things","shatter","thing"), sep="\n"))
)
canon <- function(a, b) paste(pmin(a, b), pmax(a, b), sep = "|")
focal_df <- do.call(rbind, lapply(focal_pairs, function(p) {
  key <- canon(p[[1]], p[[2]])
  row <- pair_means[pair_means$sentence_pair == key, ]
  if (nrow(row) == 0) return(NULL)
  row$label <- p[[3]]; row
}))

# Same-sense probability runs from red (0) to blue (1), for points and lines.
SS_SCALE <- scale_color_gradient(name = "Same-sense probability",
                                 low = "#d73027", high = "#4575b4",
                                 limits = c(0, 1), breaks = c(0, 0.5, 1))
make_panel <- function() {
  ggplot() +
    geom_point(data = pair_means,
               aes(x = aspect_match_j, y = similarity, color = p_same_sense),
               alpha = 0.35, size = 2.4) +
    geom_ribbon(data = smooth_match,
                aes(x = x, ymin = lo, ymax = hi, fill = p_same_sense,
                    group = interaction(p_same_sense, contpair)),
                alpha = 0.15, color = NA) +
    scale_fill_gradient(low = "#d73027", high = "#4575b4", limits = c(0, 1),
                        guide = "none") +
    geom_line(data = smooth_match,
              aes(x = x, y = y, color = p_same_sense, linetype = contpair,
                  group = interaction(p_same_sense, contpair)),
              linewidth = 1.5) +
    SS_SCALE +
    scale_linetype_manual(name = "Generality pair",
      values = c("contentful-contentful" = "solid",
                 "bleached-contentful"   = "dashed")) +
    geom_point(data = focal_df, aes(x = aspect_match, y = similarity),
               color = "black", size = 5, inherit.aes = FALSE) +
    geom_label_repel(data = focal_df,
      aes(x = aspect_match, y = similarity, label = label),
      color = "black", fill = "white", size = 4.5, fontface = "bold",
      box.padding = 0.7, point.padding = 0.3,
      min.segment.length = 0, segment.color = "black",
      max.overlaps = Inf, seed = 1, alpha = 1, inherit.aes = FALSE) +
    ylab("Dissimilarity rating") + xlab("Aspectual match") +
    scale_x_continuous(breaks = seq(0, 1, by = 0.1)) +
    coord_cartesian(ylim = c(0, 1), clip = "off") +
    guides(color = guide_colorbar(barwidth = unit(6, "cm"), order = 1,
                                  title.vjust = 0.9),
           linetype = guide_legend(order = 2)) +
    theme(legend.position = "bottom",
          legend.key.width = unit(1.5, "cm"),
          legend.title = element_text(size = 18, face = "bold",
                                      margin = margin(r = 12)),
          legend.spacing.x = unit(1, "cm"),
          legend.text  = element_text(size = 14, color = "black"),
          axis.text  = element_text(size = 18, color = "black"),
          axis.text.x = element_text(angle = 45, hjust = 1),
          axis.title = element_text(size = 22, face = "bold"))
}
p_rel <- make_panel()

# Saved at the same width and text sizes as Figures 1-3, so that the four
# figures have the same type size when set at the same width in the paper.
ggsave("figures/similarity_by_match.pdf", p_rel,
       width = 18, height = 7.5, dpi = 300)
ggsave("figures/similarity_by_match.png", p_rel,
       width = 18, height = 7.5, dpi = 300)
print(p_rel)


## §6 Discussion: responses with the adverbial held fixed

The discussion considers a weaker position than the one the paper takes: that
a bare event description lacks a fixed aspect, but that adding an *in*- or
*for*-adverbial forces one. Experiment 1b bears on this, since each of its
trials presents a sentence with one adverbial. If the adverbial fixed the
description's aspect, responses to a given sentence under a given adverbial
should tend to agree.

The cell below takes each combination of sentence and preposition in
Experiment 1b and reports (i) how many combinations received a unanimous
response on the five-point proportion scale and (ii) the share of responses
that fall on the most frequent option, averaged over combinations.

To summarize, the pipeline runs in four steps: (i) fit the cloze and proportion
models, whose random effects absorb the standard formal predictors of aspect and
whose sense weights are estimated; (ii) evaluate each sentence at its own modal
time unit to obtain posterior draws of its three logits; (iii) compute each
pair's probability that a reference sentence does not separate the two on any
logit; and (iv)
predict similarity from aspectual match, same-sense, and generality pair.


In [ ]:
# Agreement among responses to the same sentence under the same preposition
# in Experiment 1b (cited in §6).
resp_agreement <- prop_data %>%
  group_by(preposition, sentence) %>%
  summarize(n_resp      = n(),
            unanimous   = n_distinct(response) == 1,
            modal_share = max(table(response)) / n(), .groups = "drop")
print(resp_agreement %>%
  group_by(preposition) %>%
  summarize(sentences        = n(),
            mean_responses   = mean(n_resp),
            n_unanimous      = sum(unanimous),
            mean_modal_share = mean(modal_share), .groups = "drop") %>%
  as.data.frame(), digits = 3)
cat(sprintf("\nOverall: %d of %d sentence-preposition combinations unanimous; mean share on the most frequent response %.2f\n",
            sum(resp_agreement$unanimous), nrow(resp_agreement),
            mean(resp_agreement$modal_share)))
